In [38]:
import pandas as pd
import re
import csv
import re
import sys
from collections import Counter, defaultdict

df_extract = pd.read_csv(
    "../data/interim/1_2_extract_15_16_concat.csv",
    low_memory=False,
)
df_extract.shape

(1127829, 37)

In [39]:
df_ND1516 = pd.read_csv(
    "../data/raw/test_regards_citoyens/ND15+16_interventions_hemicycle_rich.tsv",
    sep="\t",
    engine="python",
    on_bad_lines="warn",
)

df_ND1516.shape

(1391207, 16)

In [40]:
df_ND1516.columns

Index(['id', 'seance_id', 'date', 'moment', 'type', 'section', 'sous_section',
       'timestamp', 'intervention', 'nb_mots', 'personnalite', 'parlementaire',
       'parlementaire_sexe', 'parlementaire_groupe', 'fonction', 'source'],
      dtype='object')

## version simple

In [41]:
# === Comparaison simple par id_syceron ===

# extraite equivalent id_syceron vs url Pnum (pas toujours le P)
P_NUMBER_RE = re.compile(r"#P?(\d+)", re.I)

def extract_pnum(url):
    if not url or pd.isna(url):
        return None
    url = str(url).strip()
    m = P_NUMBER_RE.search(url)
    return m.group(1) if m else None

# Colonnes normalisées
df_extract["id_syceron"] = df_extract["id_syceron"].dropna().astype(float).astype(int)
df_ND1516["pnum"] = df_ND1516["source"].apply(extract_pnum).dropna().astype(int)

# Comparaison sur les colonnes normalisées
ids_extract = set(df_extract["id_syceron"].dropna().astype(int))
ids_ND = set(df_ND1516["pnum"].dropna().astype(int))

common = ids_extract & ids_ND
only_extract = ids_extract - ids_ND
only_ND = ids_ND - ids_extract


print("=== RÉSUMÉ GLOBAL (par id_syceron uniquement) ===")
print(f"IDs communs              : {len(common):>10,}")
print(f"Uniquement dans extract  : {len(only_extract):>10,}")
print(f"Uniquement dans ND15-16  : {len(only_ND):>10,}")

print(f"\nTotal IDs extract : {len(ids_extract):>10,}")
print(f"Total IDs ND      : {len(ids_ND):>10,}")

# Inspection des divergences — isin() sur la colonne déjà normalisée
print("=== Dans extract mais PAS dans ND ===")
display(df_extract[df_extract["id_syceron"].isin(only_extract)].head(5))

print("=== Dans ND mais PAS dans extract ===")
display(df_ND1516[df_ND1516["pnum"].isin(only_ND)].head(5))

=== RÉSUMÉ GLOBAL (par id_syceron uniquement) ===
IDs communs              :  1,065,985
Uniquement dans extract  :     61,477
Uniquement dans ND15-16  :     43,516

Total IDs extract :  1,127,462
Total IDs ND      :  1,109,501
=== Dans extract mais PAS dans ND ===


,uid,SeanceRef,SessionRef,dateSeance,dateSeanceJour,numSeanceJour,numSeance,typeAssemblee,legislature,session,...,code_grammaire,code_style,code_parole,id_syceron,roledebat,nom_orateur,qualite_orateur,id_orateur,stime,texte
1,CRSANR5L15S2017E1N001,NaN,NaN,20170704150000000,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,OUV_SEAN_2_2,Info Italiques,NaN,981339,NaN,NaN,NaN,NaN,NaN,(La séance est ouverte à quinze heures.)
255,CRSANR5L15S2017E1N001,NaN,NaN,20170704150000000,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,SUSP_SEANCE_2_2,Info Italiques,NaN,982051,NaN,NaN,NaN,NaN,NaN,"(La séance, suspendue à dix-huit heures quinze..."
259,CRSANR5L15S2017E1N001,NaN,NaN,20170704150000000,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,FIN_SEAN_2_4,Info Italiques,NaN,982060,NaN,NaN,NaN,NaN,NaN,(La séance est levée à dix-huit heures cinquan...
260,CRSANR5L15S2017E1N001,NaN,NaN,20170704150000000,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,FIN_SEAN_2_4,Info Italiques,NaN,982060,NaN,NaN,NaN,NaN,NaN,La Directrice du service du compte rendu de la...
262,CRSANR5L15S2017E1N002,NaN,NaN,20170705150000000,mercredi 05 juillet 2017,Unique,2,AN,15,Première session extraordinaire 2017,...,OUV_SEAN_2_2,Info Italiques,NaN,982176,NaN,NaN,NaN,NaN,NaN,(La séance est ouverte à quinze heures.)


=== Dans ND mais PAS dans extract ===


,id,seance_id,date,moment,type,section,sous_section,timestamp,intervention,nb_mots,personnalite,parlementaire,parlementaire_sexe,parlementaire_groupe,fonction,source,pnum
2,3,1,2017-06-27,15:00,loi,ouverture de la xve législature,ouverture de la xve législature,60,<p>Ouverture de la XVe législature</p>,8,NaN,NaN,NaN,NaN,NaN,http://www.assemblee-nationale.fr/15/cri/2016-...,980119.0
4,5,1,2017-06-27,15:00,loi,constitution du bureau d'âge,constitution du bureau d'âge,80,<p>constitution du bureau d'âge</p>,7,NaN,NaN,NaN,NaN,NaN,http://www.assemblee-nationale.fr/15/cri/2016-...,980122.0
6,7,1,2017-06-27,15:00,loi,communication de la liste des députés,communication de la liste des députés,100,<p>communication de la liste des députés</p>,10,NaN,NaN,NaN,NaN,NaN,http://www.assemblee-nationale.fr/15/cri/2016-...,980125.0
8,9,1,2017-06-27,15:00,loi,députés nommés membres du gouvernement,députés nommés membres du gouvernement,120,<p>députés nommés membres du gouvernement</p>,10,NaN,NaN,NaN,NaN,NaN,http://www.assemblee-nationale.fr/15/cri/2016-...,980128.0
10,11,1,2017-06-27,15:00,loi,décès de deux députés de la xive législature,décès de deux députés de la xive législature,140,<p>décès de deux députés de la xive législatur...,15,NaN,NaN,NaN,NaN,NaN,http://www.assemblee-nationale.fr/15/cri/2016-...,980131.0


In [42]:
# Vérification sur un échantillon aléatoire des IDs communs
sample_common = pd.Series(list(common)).sample(10)

df_sample_extract = df_extract[df_extract["id_syceron"].isin(sample_common)][
    ["id_syceron", "texte"]
].rename(columns={"id_syceron": "id"})  # ← adapter si la colonne texte a un autre nom

df_sample_ND = df_ND1516[df_ND1516["pnum"].isin(sample_common)][
    ["pnum", "intervention"]
].rename(columns={"pnum": "id"})  # ← adapter si la colonne intervention a un autre nom

# Fusion sur l'id commun
df_check = df_sample_extract.merge(df_sample_ND, on="id")

display(df_check)

print(
    "LÉO, NORMAL QUE TU AIES DES 'DOUBLONS'",
    "\nTON FICHIER MARCHE C'EST À CAUSE  DU CHANGEMENT TEXTE DANS ND)",
)

,id,texte,intervention
0,1058190,"Pour ma part, en tout cas, je ne vous accuse p...","<p>Pour ma part, en tout cas, je ne vous accus..."
1,1653316,"La parole est à Mme Véronique Louwagie, pour s...","<p>La parole est à Mme Véronique Louwagie, pou..."
2,1694008,À la suite du braquage de Pénélope !,<p>À la suite du braquage de Pénélope !</p>
3,2160532,Les deux projets de loi que vous nous proposez...,<p>Les deux projets de loi que vous nous propo...
4,2065190,Je laisse le Gouvernement répondre concernant ...,<p>Je laisse le Gouvernement répondre concerna...
5,2629562,"La parole est à M. Hervé Pellois, pour souteni...","<p>La parole est à M. Hervé Pellois, pour sout..."
6,2841188,"L’objectif, c’était surtout la réélection d’Em...","<p>L'objectif, c'était surtout la réélection d..."
7,3093317,Quel est l’avis de la commission ?,<p>Quel est l'avis de la commission ?</p>
8,3288962,"Madame Genevard, avons-nous réellement tiré le...","<p>Madame Genevard, avons-nous réellement tiré..."
9,3288962,"Madame Genevard, avons-nous réellement tiré le...",<p>Exclamations sur les bancs du groupe RN</p>


LÉO, NORMAL QUE TU AIES DES 'DOUBLONS' 
TON FICHIER MARCHE C'EST À CAUSE  DU CHANGEMENT TEXTE DANS ND)


In [43]:
df_only_ND = df_ND1516[df_ND1516["pnum"].isin(only_ND)]
df_only_ND

,id,seance_id,date,moment,type,section,sous_section,timestamp,intervention,nb_mots,personnalite,parlementaire,parlementaire_sexe,parlementaire_groupe,fonction,source,pnum
2,3,1,2017-06-27,15:00,loi,ouverture de la xve législature,ouverture de la xve législature,60,<p>Ouverture de la XVe législature</p>,8,NaN,NaN,NaN,NaN,NaN,http://www.assemblee-nationale.fr/15/cri/2016-...,980119.0
4,5,1,2017-06-27,15:00,loi,constitution du bureau d'âge,constitution du bureau d'âge,80,<p>constitution du bureau d'âge</p>,7,NaN,NaN,NaN,NaN,NaN,http://www.assemblee-nationale.fr/15/cri/2016-...,980122.0
6,7,1,2017-06-27,15:00,loi,communication de la liste des députés,communication de la liste des députés,100,<p>communication de la liste des députés</p>,10,NaN,NaN,NaN,NaN,NaN,http://www.assemblee-nationale.fr/15/cri/2016-...,980125.0
8,9,1,2017-06-27,15:00,loi,députés nommés membres du gouvernement,députés nommés membres du gouvernement,120,<p>députés nommés membres du gouvernement</p>,10,NaN,NaN,NaN,NaN,NaN,http://www.assemblee-nationale.fr/15/cri/2016-...,980128.0
10,11,1,2017-06-27,15:00,loi,décès de deux députés de la xive législature,décès de deux députés de la xive législature,140,<p>décès de deux députés de la xive législatur...,15,NaN,NaN,NaN,NaN,NaN,http://www.assemblee-nationale.fr/15/cri/2016-...,980131.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1390748,608261,3000,2024-06-07,21:30,loi,accompagnement des malades et de la fin de vie...,article 7,1430,<p>Sourires.</p>,2,NaN,NaN,NaN,NaN,NaN,https://www.assemblee-nationale.fr/16/cri/2023...,3507002.0
1390843,608356,3000,2024-06-07,21:30,loi,accompagnement des malades et de la fin de vie...,discussion des articles,2380,<p>Rappel au règlement</p>,5,NaN,NaN,NaN,NaN,NaN,https://www.assemblee-nationale.fr/16/cri/2023...,3507238.0
1390857,608370,3000,2024-06-07,21:30,loi,accompagnement des malades et de la fin de vie...,article 7,2520,<p>Article 7</p>,2,NaN,NaN,NaN,NaN,NaN,https://www.assemblee-nationale.fr/16/cri/2023...,3507257.0
1390898,608411,3000,2024-06-07,21:30,loi,accompagnement des malades et de la fin de vie...,article 7,2930,<p>Nous discutons d'un projet de loi pour des ...,46,NaN,Laurence Maillart-Méhaignerie,F,REN,rapporteure,https://www.assemblee-nationale.fr/16/cri/2023...,3507303.0


In [44]:
# Vérifie les cas où ni "parlementaire" ni "personnalite" ne sont renseignés
mask_no_speaker = (
    df_only_ND["parlementaire"].fillna("").astype(str).str.strip().eq("")
    & df_only_ND["personnalite"].fillna("").astype(str).str.strip().eq("")
)

df_no_speaker = df_only_ND[mask_no_speaker]

print(f"Lignes sans parlementaire ET sans personnalite : {len(df_no_speaker):,} / {len(df_only_ND):,}")
display(df_no_speaker.head(20))

Lignes sans parlementaire ET sans personnalite : 44,570 / 45,408


,id,seance_id,date,moment,type,section,sous_section,timestamp,intervention,nb_mots,personnalite,parlementaire,parlementaire_sexe,parlementaire_groupe,fonction,source,pnum
2,3,1,2017-06-27,15:00,loi,ouverture de la xve législature,ouverture de la xve législature,60,<p>Ouverture de la XVe législature</p>,8,NaN,NaN,NaN,NaN,NaN,http://www.assemblee-nationale.fr/15/cri/2016-...,980119.0
4,5,1,2017-06-27,15:00,loi,constitution du bureau d'âge,constitution du bureau d'âge,80,<p>constitution du bureau d'âge</p>,7,NaN,NaN,NaN,NaN,NaN,http://www.assemblee-nationale.fr/15/cri/2016-...,980122.0
6,7,1,2017-06-27,15:00,loi,communication de la liste des députés,communication de la liste des députés,100,<p>communication de la liste des députés</p>,10,NaN,NaN,NaN,NaN,NaN,http://www.assemblee-nationale.fr/15/cri/2016-...,980125.0
8,9,1,2017-06-27,15:00,loi,députés nommés membres du gouvernement,députés nommés membres du gouvernement,120,<p>députés nommés membres du gouvernement</p>,10,NaN,NaN,NaN,NaN,NaN,http://www.assemblee-nationale.fr/15/cri/2016-...,980128.0
10,11,1,2017-06-27,15:00,loi,décès de deux députés de la xive législature,décès de deux députés de la xive législature,140,<p>décès de deux députés de la xive législatur...,15,NaN,NaN,NaN,NaN,NaN,http://www.assemblee-nationale.fr/15/cri/2016-...,980131.0
12,13,1,2017-06-27,15:00,loi,allocution du doyen d'âge,allocution du doyen d'âge,160,<p>allocution du doyen d'âge</p>,7,NaN,NaN,NaN,NaN,NaN,http://www.assemblee-nationale.fr/15/cri/2016-...,980134.0
15,16,1,2017-06-27,15:00,loi,Élection du président de l'assemblée nationale,Élection du président de l'assemblée nationale,210,<p>Élection du président de l'Assemblée nation...,10,NaN,NaN,NaN,NaN,NaN,http://www.assemblee-nationale.fr/15/cri/2016-...,980141.0
38,39,1,2017-06-27,15:00,loi,allocution de m. le président,allocution de m. le président,510,<p>Allocution de M. le président</p>,8,NaN,NaN,NaN,NaN,NaN,http://www.assemblee-nationale.fr/15/cri/2016-...,980164.0
40,41,1,2017-06-27,15:00,loi,allocution de m. le président,allocution de m. le président,530,<p>Applaudissements.</p>,3,NaN,NaN,NaN,NaN,NaN,http://www.assemblee-nationale.fr/15/cri/2016-...,980169.0
47,48,1,2017-06-27,15:00,loi,ordre du jour de la prochaine séance,ordre du jour de la prochaine séance,620,<p>Ordre du jour de la prochaine séance</p>,10,NaN,NaN,NaN,NaN,NaN,http://www.assemblee-nationale.fr/15/cri/2016-...,980214.0


In [45]:
# creuser les derniers ca qui seraient pas en missing et qui pourarient être vrai écart

In [46]:
# et chelou pour volume extract seul
df_only_extract = df_extract[df_extract["id_syceron"].isin(only_extract)]
df_only_extract

,uid,SeanceRef,SessionRef,dateSeance,dateSeanceJour,numSeanceJour,numSeance,typeAssemblee,legislature,session,...,code_grammaire,code_style,code_parole,id_syceron,roledebat,nom_orateur,qualite_orateur,id_orateur,stime,texte
1,CRSANR5L15S2017E1N001,NaN,NaN,20170704150000000,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,OUV_SEAN_2_2,Info Italiques,NaN,981339,NaN,NaN,NaN,NaN,NaN,(La séance est ouverte à quinze heures.)
255,CRSANR5L15S2017E1N001,NaN,NaN,20170704150000000,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,SUSP_SEANCE_2_2,Info Italiques,NaN,982051,NaN,NaN,NaN,NaN,NaN,"(La séance, suspendue à dix-huit heures quinze..."
259,CRSANR5L15S2017E1N001,NaN,NaN,20170704150000000,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,FIN_SEAN_2_4,Info Italiques,NaN,982060,NaN,NaN,NaN,NaN,NaN,(La séance est levée à dix-huit heures cinquan...
260,CRSANR5L15S2017E1N001,NaN,NaN,20170704150000000,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,FIN_SEAN_2_4,Info Italiques,NaN,982060,NaN,NaN,NaN,NaN,NaN,La Directrice du service du compte rendu de la...
262,CRSANR5L15S2017E1N002,NaN,NaN,20170705150000000,mercredi 05 juillet 2017,Unique,2,AN,15,Première session extraordinaire 2017,...,OUV_SEAN_2_2,Info Italiques,NaN,982176,NaN,NaN,NaN,NaN,NaN,(La séance est ouverte à quinze heures.)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1126740,CRSANR5L16S2024O1N234,RUANR5L16S2024IDS28427,SCR5A2024O1,20240607150000000,vendredi 07 juin 2024,2,234,AN,16,Session ordinaire 2023-2024,...,DISC_ARTICLES_3_9_1,NORMAL,NaN,3506047,president,Mme la présidente,NaN,721908.0,6894.86,"Je suis saisie de trois amendements, nos 3194,..."
1127123,CRSANR5L16S2024O1N234,RUANR5L16S2024IDS28427,SCR5A2024O1,20240607150000000,vendredi 07 juin 2024,2,234,AN,16,Session ordinaire 2023-2024,...,DISC_ARTICLES_3_30,NORMAL,NaN,3507393,president,Mme la présidente,NaN,721908.0,14871.08,Nous en venons à une nouvelle série d’amendeme...
1127400,CRSANR5L16S2024O1N235,RUANR5L16S2024IDS28428,SCR5A2024O1,20240607213000000,vendredi 07 juin 2024,3,235,AN,16,Session ordinaire 2023-2024,...,PAROLE_GENERIQUE,NORMAL,NaN,3506662,NaN,Mme Laurence Cristol,rapporteure,793876.0,2767.49,Cependant il ne s’agit pas seulement de repére...
1127446,CRSANR5L16S2024O1N235,RUANR5L16S2024IDS28428,SCR5A2024O1,20240607213000000,vendredi 07 juin 2024,3,235,AN,16,Session ordinaire 2023-2024,...,DISC_ARTICLES_3_9,NORMAL,NaN,3507003,NaN,M. Thibault Bazin,NaN,642847.0,3551.97,Nous souhaitons que la décision d’accorder ou ...


In [47]:
df_only_extract_sub = df_only_extract[df_only_extract["code_style"] == "NORMAL"]

In [48]:
df_only_extract_sub["nom_orateur"].value_counts()

nom_orateur
Mme la présidente             207
M. le président               142
Mme Geneviève Darrieussecq     17
Mme Élisabeth Borne            16
M. Thibault Bazin              15
                             ... 
M. Paul Molac                   1
M. Julien Ravier                1
M. Lionel Causse                1
M. Alexis Corbière,             1
M. Michel Lauzzana              1
Name: count, Length: 338, dtype: int64

In [49]:
df_only_extract[df_only_extract["nom_orateur"] == "M. Ugo Bernalicis"]

,uid,SeanceRef,SessionRef,dateSeance,dateSeanceJour,numSeanceJour,numSeance,typeAssemblee,legislature,session,...,code_grammaire,code_style,code_parole,id_syceron,roledebat,nom_orateur,qualite_orateur,id_orateur,stime,texte
602253,CRSANR5L15S2021O1N084,NaN,NaN,20201119090000000,jeudi 19 novembre 2020,1,84,AN,15,session ordinaire 2020-2021,...,INTERRUPTION_1_10,NORMAL,NaN,2322561,NaN,M. Ugo Bernalicis,NaN,720430.0,NaN,On peut même dire qu’ils étaient tous d’accord !
602430,CRSANR5L15S2021O1N084,NaN,NaN,20201119090000000,jeudi 19 novembre 2020,1,84,AN,15,session ordinaire 2020-2021,...,INTERRUPTION_1_10,NORMAL,NaN,2322566,NaN,M. Ugo Bernalicis,NaN,720430.0,NaN,Il n’y a pas que moi qui le dis !
602532,CRSANR5L15S2021O1N084,NaN,NaN,20201119090000000,jeudi 19 novembre 2020,1,84,AN,15,session ordinaire 2020-2021,...,INTERRUPTION_1_10,NORMAL,NaN,2322569,NaN,M. Ugo Bernalicis,NaN,720430.0,NaN,"Un petit sophisme en réponse, monsieur le mini..."
602613,CRSANR5L15S2021O1N084,NaN,NaN,20201119090000000,jeudi 19 novembre 2020,1,84,AN,15,session ordinaire 2020-2021,...,INTERRUPTION_1_10,NORMAL,NaN,2322570,NaN,M. Ugo Bernalicis,NaN,720430.0,NaN,Exactement ! Je suis d’accord !
627067,CRSANR5L15S2021O1N130,NaN,NaN,20210114090000000,jeudi 14 janvier 2021,1,130,AN,15,session ordinaire 2020-2021,...,PAROLE_GENERIQUE,NORMAL,PAROLE_1_2,2370998,NaN,M. Ugo Bernalicis,NaN,720430.0,NaN,"Enfin, le Conseil supérieur de la magistrature..."
636579,CRSANR5L15S2021O1N153,NaN,NaN,20210205090000000,vendredi 05 février 2021,1,153,AN,15,session ordinaire 2020-2021,...,INTERRUPTION_1_10,NORMAL,NaN,2397115,NaN,M. Ugo Bernalicis,NaN,720430.0,NaN,Eh oui !
636621,CRSANR5L15S2021O1N153,NaN,NaN,20210205090000000,vendredi 05 février 2021,1,153,AN,15,session ordinaire 2020-2021,...,INTERRUPTION_1_10,NORMAL,NaN,2397118,NaN,M. Ugo Bernalicis,NaN,720430.0,NaN,Ils ne doutent vraiment de rien !
800582,CRSANR5L16S2022E1N015,RUANR5L16S2022IDS26212,SCR5A2022E1,20220721213000000,jeudi 21 juillet 2022,3,15,AN,16,Première session extraordinaire 2022,...,INTERRUPTION_1_10,NORMAL,NaN,2834757,NaN,M. Ugo Bernalicis,NaN,720430.0,30101.99,Mme Le Pen est la candidate non du pouvoir d’a...


In [50]:
test = df_ND1516[df_ND1516["intervention"].str.contains("Soixante-quatre ans ou pas ?")]
test

,id,seance_id,date,moment,type,section,sous_section,timestamp,intervention,nb_mots,personnalite,parlementaire,parlementaire_sexe,parlementaire_groupe,fonction,source,pnum
841415,1188109,8407,2021-07-06,15:00,question,questions au gouvernement > réforme des retraites,réforme des retraites,2060,<p>Soixante-quatre ans ou pas ? Vous n'avez pa...,11,NaN,Ugo Bernalicis,H,LFI,NaN,http://www.assemblee-nationale.fr/15/cri/2020-...,2576326.0


In [51]:
df_extract

,uid,SeanceRef,SessionRef,dateSeance,dateSeanceJour,numSeanceJour,numSeance,typeAssemblee,legislature,session,...,code_grammaire,code_style,code_parole,id_syceron,roledebat,nom_orateur,qualite_orateur,id_orateur,stime,texte
0,CRSANR5L15S2017E1N001,NaN,NaN,20170704150000000,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,OUV_SEAN_2_1,NORMAL,NaN,981338,president,M. le président,NaN,332747.0,NaN,La séance est ouverte.
1,CRSANR5L15S2017E1N001,NaN,NaN,20170704150000000,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,OUV_SEAN_2_2,Info Italiques,NaN,981339,NaN,NaN,NaN,NaN,NaN,(La séance est ouverte à quinze heures.)
2,CRSANR5L15S2017E1N001,NaN,NaN,20170704150000000,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,ODJ_APPEL_DISCUSSION,NORMAL,NaN,981342,president,M. le président,NaN,332747.0,NaN,En application des articles 29 et 30 de la Con...
3,CRSANR5L15S2017E1N001,NaN,NaN,20170704150000000,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,ODJ_APPEL_DISCUSSION,NORMAL,NaN,981345,president,M. le président,NaN,332747.0,NaN,L’ordre du jour appelle la déclaration de poli...
4,CRSANR5L15S2017E1N001,NaN,NaN,20170704150000000,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,DEBAT_1_10,NORMAL,PAROLE_1_2,981347,NaN,M. Edouard Philippe,Premier ministre,345619.0,NaN,"Monsieur le président, mesdames, messieurs les..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1127824,CRSANR5L16S2024O1N235,RUANR5L16S2024IDS28428,SCR5A2024O1,20240607213000000,vendredi 07 juin 2024,3,235,AN,16,Session ordinaire 2023-2024,...,SCRUT_ADTS_1_9,Info Italiques,NaN,3507788,NaN,NaN,NaN,NaN,NaN,(Les amendements identiques nos 518 et 2220 ne...
1127825,CRSANR5L16S2024O1N235,RUANR5L16S2024IDS28428,SCR5A2024O1,20240607213000000,vendredi 07 juin 2024,3,235,AN,16,Session ordinaire 2023-2024,...,FIN_SEAN_1_0,NORMAL,NaN,3507835,president,Mme la présidente,NaN,721908.0,9358.93,La suite de la discussion est renvoyée à la pr...
1127826,CRSANR5L16S2024O1N235,RUANR5L16S2024IDS28428,SCR5A2024O1,20240607213000000,vendredi 07 juin 2024,3,235,AN,16,Session ordinaire 2023-2024,...,FIN_SEAN_2_1,NORMAL,NaN,3507792,president,Mme la présidente,NaN,721908.0,9380.21,"Prochaine séance, lundi, à quinze heures : Sui..."
1127827,CRSANR5L16S2024O1N235,RUANR5L16S2024IDS28428,SCR5A2024O1,20240607213000000,vendredi 07 juin 2024,3,235,AN,16,Session ordinaire 2023-2024,...,FIN_SEAN_2_4,Info Italiques,NaN,3507793,NaN,NaN,NaN,NaN,NaN,(La séance est levée à vingt-trois heures cinq...


In [52]:
## Version avec commbinaison ID et date

In [53]:
# FILE1_DATE_COL  = "dateSeance"   # format 20170704150000000
# FILE1_KEY_COL   = "id_syceron"
# FILE2_DATE_COL  = "date"         # format 2017-07-04
# FILE2_SOURCE_COL = "source"      # contient ...#P980116

# P_NUMBER_RE = re.compile(r"#P(\d+)")

# # extraite equivalent id_syceron vs url Pnum
# def extract_pnum(url):
#     if not url:
#         return None
#     m = P_NUMBER_RE.search(url)
#     return m.group(1) if m else None

# # === Normalisation des dates ===

# def normaliser_date(raw):
#     """Ramène tout format de date à YYYYMMDD (str 8 chars)."""
#     if pd.isna(raw):
#         return "UNKNOWN"
#     raw = str(raw).strip()
#     # format 20170704150000000 ou 20170704
#     raw = raw.replace("-", "")  # format 2017-07-04 → 20170704
#     return raw[:8]

# df_extract["dateSeance"] = df_extract["dateSeance"].apply(normaliser_date)
# df_ND1516["date"]  = df_ND1516["date"].apply(normaliser_date)


# # === Construction des index de clés ===

# def build_keys(df, date_col, key_extractor):
#     """
#     Construit les index de clés à partir d'un DataFrame déjà chargé.
#     Retourne :
#       - rows_per_date  : Counter(date -> nb lignes)
#       - keys_per_date  : dict(date -> set(cle))
#       - all_keys       : set((date, cle))
#     """
#     rows_per_date = Counter()
#     keys_per_date = defaultdict(set)
#     all_keys = set()

#     for _, row in df.iterrows():
#         raw_date = str(row.get(date_col, "") or "")
#         date = raw_date.replace("-", "")[:8] if raw_date else "UNKNOWN"
#         rows_per_date[date] += 1
#         key = key_extractor(row)
#         if key:
#             keys_per_date[date].add(key)
#             all_keys.add((date, key))

#     return rows_per_date, keys_per_date, all_keys

# print("Construction des index...")
# rows1, keys1, all_keys1 = build_keys(
#     df_extract,
#     FILE1_DATE_COL,
#     lambda r: r.get(FILE1_KEY_COL) or None,
# )

# rows2, keys2, all_keys2 = build_keys(
#     df_ND1516,
#     FILE2_DATE_COL,
#     lambda r: extract_pnum(r.get(FILE2_SOURCE_COL)),
# )

# print(f"df_extract : {len(df_extract):,} lignes | {len(all_keys1):,} unités (date+id) uniques")
# print(f"df_ND1516  : {len(df_ND1516):,} lignes | {len(all_keys2):,} unités (date+id) uniques")



In [54]:
# # === Comparaison globale ===

# common = all_keys1 & all_keys2
# only1  = all_keys1 - all_keys2
# only2  = all_keys2 - all_keys1

# print("=== RÉSUMÉ GLOBAL ===")
# print(f"Unités communes                : {len(common):>10,}")
# print(f"Uniquement dans extract        : {len(only1):>10,}")
# print(f"Uniquement dans ND15-16        : {len(only2):>10,}")

In [55]:
# # === Détail par jour ===

# all_dates = sorted(set(rows1) | set(rows2))
# print(f"{'date':10} {'f1_lignes':>10} {'f1_unites':>10} {'f2_lignes':>10} {'f2_unites':>10} {'ecart':>8}")
# for d in all_dates:
#     r1 = rows1.get(d, 0)
#     u1 = len(keys1.get(d, set()))
#     r2 = rows2.get(d, 0)
#     u2 = len(keys2.get(d, set()))
#     ecart = u2 - u1
#     flag = "  ← absent f1" if r1 == 0 else ("  ← absent f2" if r2 == 0 else "")
#     print(f"{d:10} {r1:>10,} {u1:>10,} {r2:>10,} {u2:>10,} {ecart:>8}{flag}")